In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-class-path /Users/sydneysailors/smart-store-sydney/lib/sqlite-jdbc-3.51.0.0.jar "
    "--jars /Users/sydneysailors/smart-store-sydney/lib/sqlite-jdbc-3.51.0.0.jar "
    "pyspark-shell"
)

In [2]:
from pyspark.sql import SparkSession

dw_path = "/Users/sydneysailors/smart-store-sydney/data/dw/smart_sales.db"

spark = (
    SparkSession.builder
    .appName("SmartStoreReporting")
    .getOrCreate()
)

# Test driver
spark._jvm.java.lang.Class.forName("org.sqlite.JDBC")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 16:11:32 WARN Utils: Your hostname, Sydneys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.215.82.145 instead (on interface en0)
25/11/23 16:11:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/11/23 16:11:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


JavaObject id=o31

In [3]:
sales_df = spark.read.format("jdbc").options(
    url=f"jdbc:sqlite:{dw_path}",
    dbtable="sales",
    driver="org.sqlite.JDBC"
).load()

sales_df.show(5)

+-------+---------+-----------+----------+--------+-----------+------------+---------------+-----+
|sale_id|sale_date|customer_id|product_id|store_id|campaign_id|sales_amount|discount_amount|state|
+-------+---------+-----------+----------+--------+-----------+------------+---------------+-----+
|      1|   5/4/25|       1034|      2059|     402|        0.0|      2048.2|          12.45|   CA|
|      2|   5/4/25|       1066|      2048|     403|        1.0|      321.87|            7.8|   NY|
|      3|   5/4/25|       1116|      2041|     403|        3.0|     3216.84|           25.0|   TX|
|      4|   5/4/25|       1071|      2096|     404|        2.0|     1613.23|           18.3|   FL|
|      5|   5/4/25|       1020|      2060|     401|        0.0|      408.38|            5.5|   IL|
+-------+---------+-----------+----------+--------+-----------+------------+---------------+-----+
only showing top 5 rows


In [6]:
customers_df = spark.read.format("jdbc").options(
    url=f"jdbc:sqlite:{dw_path}",
    dbtable="customers",
    driver="org.sqlite.JDBC"
).load()

products_df = spark.read.format("jdbc").options(
    url=f"jdbc:sqlite:{dw_path}",
    dbtable="products",
    driver="org.sqlite.JDBC"
).load()

customers_df.show(5)
products_df.show(5)

+-----------+-------------+-------+---------+---+-------------------+
|customer_id|         name| region|join_date|age|subscription_status|
+-----------+-------------+-------+---------+---+-------------------+
|       1000| Robert Gomez|   West|  2/25/24| 49|            Expired|
|       1001|   John Silva|   East|  12/1/20| 48|              Trial|
|       1002|Mark Marshall|Central|   8/8/20| 37|              Trial|
|       1003|David Brennan|  North|  5/21/20| 71|              Trial|
|       1004|Kerry Collins|  North|  9/12/23| 61|            Expired|
+-----------+-------------+-------+---------+---+-------------------+
only showing top 5 rows
+----------+--------------------+-----------+----------+----------------+-------------------+
|product_id|        product_name|   category|unit_price|manufacture_year|availability_status|
+----------+--------------------+-----------+----------+----------------+-------------------+
|      2000|      Electronics-Be|Electronics|     969.0|        

In [8]:
sales_df.createOrReplaceTempView("sale")
customers_df.createOrReplaceTempView("customer")
products_df.createOrReplaceTempView("product")